<a href="https://colab.research.google.com/github/JamesMartinOU/PublicRedditSentimentAnalysis/blob/main/RedditCreateSentimentFactTable__YearMonth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Python libraries
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.0/34.0 MB 12.3 MB/s eta 0:00:00


In [3]:
# Import Python libraries
import mysql.connector
import pandas as pd
from google.colab import files

In [4]:
# RDS MySQL connection details
DB_HOST = "redditdb.cfk6syyo42ze.us-east-2.rds.amazonaws.com"
DB_USER = "JamesMartinOU"  # Your MySQL username
DB_PASSWORD = "Bedard99*"  # Your MySQL password
DB_NAME = "redditdb"  # Your database name

In [7]:
# Connect to the MySQL database
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

# Query to create sentiment fact table
query_create_fact_table = """
CREATE TABLE reddit_sentiment_fact_table (
  id INT AUTO_INCREMENT PRIMARY KEY,
  type VARCHAR(10),
  company_name VARCHAR(255),
  keyword VARCHAR(255),
  keyword_label VARCHAR(255),
  year INT,
  month INT,
  sentiment VARCHAR(20),
  count INT
);
"""

# Query to generate sentiment fact table data
query_fact_table_sentiment = """
INSERT INTO
  reddit_sentiment_fact_table (type, company_name, keyword, keyword_label, year, month, sentiment, count)

SELECT
  "comment" as type,
  rcsgb.company_name,
  rcsgb.keyword,
  CASE
    WHEN rcsgb.keyword = rcsgb.company_name THEN "company_name"
    WHEN length(rcsgb.keyword) <= 5 THEN "stock_symbol"
    ELSE "ceo_name"
  END as keyword_label,
  year(rcsgb.created_utc) as year,
  month(rcsgb.created_utc) as month,
  rcsgb.avg_sentiment as sentiment,
  count(rcsgb.comment_id) as count

FROM (
  SELECT
    arc.post_id,
    arc.comment_id,
    arc.created_utc,
    rp.company_name,
    rp.keyword,
    CASE
      WHEN avg(arc.sentiment_score) >= .33 THEN "Positive"
      WHEN avg(arc.sentiment_score) <= -.33 THEN "Negative"
      ELSE "Neutral"
    END as avg_sentiment
  FROM (
    SELECT
      rcs.post_id,
      rcs.comment_id,
      rc.created_utc,
      CASE
        WHEN rcs.sentiment = "Neutral" THEN 0
        WHEN rcs.sentiment = "Positive" THEN 1
        ELSE -1
      END as sentiment_score
    FROM reddit_comments_sentiment as rcs
    INNER JOIN reddit_comments as rc ON rc.comment_id = rcs.comment_id
  ) as arc
  INNER JOIN
    reddit_posts as rp ON rp.id = arc.post_id
  GROUP BY
    arc.post_id,
    arc.comment_id,
    arc.created_utc,
    rp.company_name,
    rp.keyword
    ) as rcsgb

GROUP BY
  rcsgb.company_name,
  rcsgb.keyword,
  keyword_label,
  year(rcsgb.created_utc),
  month(rcsgb.created_utc),
  rcsgb.avg_sentiment

UNION ALL

SELECT
  "post" as type,
  rp.Company_Name,
  rp.Keyword,
  CASE
    WHEN rp.keyword = rp.company_name THEN "company_name"
    WHEN length(rp.keyword) <= 5 THEN "stock_symbol"
    ELSE "ceo_name"
  END as keyword_label,
  year(created_utc) as year,
  month(created_utc) as month,
  rps.Sentiment,
  count(rp.id) as count

FROM reddit_posts as rp
INNER JOIN
  reddit_posts_sentiment as rps ON rps.id = rp.id
GROUP BY
  rp.Company_Name,
  rp.Keyword,
  keyword_label,
  year(created_utc),
  month(created_utc),
  rps.Sentiment
"""


# Create cursor and execute
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS reddit_sentiment_fact_table")
cursor.execute(query_create_fact_table)
cursor.execute(query_fact_table_sentiment)

# View results
cursor.execute("SELECT * FROM reddit_sentiment_fact_table")
columns = [desc[0] for desc in cursor.description]
data = cursor.fetchall()
df_sentiment_fact = pd.DataFrame(data, columns=columns)
print(df_sentiment_fact.head())

# Cleanup
cursor.close()
conn.close()


   id     type company_name keyword keyword_label  year  month sentiment  \
0   1  comment       AbbVie    ABBV  stock_symbol  2016      2  Negative   
1   2  comment       AbbVie    ABBV  stock_symbol  2016      2   Neutral   
2   3  comment       AbbVie    ABBV  stock_symbol  2016      2  Positive   
3   4  comment       AbbVie    ABBV  stock_symbol  2019     12  Negative   
4   5  comment       AbbVie    ABBV  stock_symbol  2019     12   Neutral   

   count  
0      3  
1      3  
2      4  
3      4  
4      3  
